# PASO 1 - INSTALAMOS LAS DEPENDENCIAS DE HUGGING FACE

In [1]:
!pip install datasets transformers evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and pl

# PASO 2 - IMPORTAMOS EL DATASET

In [2]:
from datasets import load_dataset
ds = load_dataset("glue","mrpc")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [4]:
ds['train'][100]

{'sentence1': 'The Nasdaq composite index inched up 1.28 , or 0.1 percent , to 1,766.60 , following a weekly win of 3.7 percent .',
 'sentence2': 'The technology-laced Nasdaq Composite Index .IXIC was off 24.44 points , or 1.39 percent , at 1,739.87 .',
 'label': 0,
 'idx': 114}

In [5]:
ds['train'].features['label']

ClassLabel(names=['not_equivalent', 'equivalent'], id=None)

# PASO 3 - TOKENIZADO CON MODELOS DE HUGGING FACE

In [6]:
from transformers import AutoTokenizer

repo_id = 'distilroberta-base'

tokenizer = AutoTokenizer.from_pretrained(repo_id)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [7]:
tokenized_sentence_1 = tokenizer(ds['train']['sentence1'][2])
tokenized_sentence_1

{'input_ids': [0, 1213, 56, 1027, 41, 6859, 15, 5, 3742, 15, 502, 158, 2156, 1839, 5, 9145, 13, 1392, 2156, 37, 355, 479, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [8]:
def tokenize_fn(texts):
  return tokenizer(texts['sentence1'],texts['sentence2'],truncation=True)

prepared_ds = ds.map(tokenize_fn,batched=True)
prepared_ds

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [9]:
#agregamos padding
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# PASO 4 - CONFIGURAMOS EL MODELO

In [10]:
from transformers import AutoModelForSequenceClassification

labels = ds['train'].features['label'].names

model = AutoModelForSequenceClassification.from_pretrained(
    repo_id,
    num_labels=len(labels),
    id2label={str(i): c for i,c in enumerate(labels)},
    label2id={c: str(i) for i,c in enumerate(labels)}
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# PASO 5 - SUBIR EL MODELO A UN REPOSITORIO DE HUGGING FACES

## LOGIN A HUGGING FACE

In [11]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `datag3` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `datag3`


## CREAMOS ARGUMENTOS PARA EL TRANSFORMER

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir ='./JavierCh86-distilrogerta-jchacnamadatag3',
    hub_model_id='JavierCh86/distilrogerta-jchacnamadatag3',
    eval_strategy='steps',
    num_train_epochs=3,
    push_to_hub=True,
    load_best_model_at_end=True,
)

## CREAMOS FUNCIÓN PARA METRICAS

In [14]:
import evaluate
import numpy as np

def compute_metrics(eval_pred):
  metric = evaluate.load('glue','mrpc')
  logits,labels = eval_pred
  predictions = np.argmax(logits,axis=-1)
  return metric.compute(predictions=predictions,references=labels)

## CONFIGURAMOS EL TRAINER DEL MODELO

In [15]:
from transformers import Trainer

trainer = Trainer(
  model,
  training_args,
  train_dataset=prepared_ds['train'],
  eval_dataset=prepared_ds['validation'],
  data_collator=data_collator,
  tokenizer=tokenizer,
  compute_metrics=compute_metrics
)

<ipython-input-15-d51073d4ec85>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## ENTRENAMOS EL MODELO

In [16]:
train_results = trainer.train()
trainer.save_model()
trainer.log_metrics("train",train_results.metrics)
trainer.save_metrics("train",train_results.metrics)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jchacnama-q (jchacnama-q-datag3) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy,F1
500,0.507600,0.585176,0.833333,0.881119
1000,0.315400,0.669203,0.850490,0.890485


wandb: 502 encountered (
wandb: <html><head>
wandb: <meta http-equiv="content-type" content="text/html;charset=utf-8">
wandb: <title>502 Server Error</title>
wandb: </head>
wandb: <body text=#000000 bgcolor=#ffffff>
wandb: <h1>Error: Server Error</h1>
wandb: <h2>The server encountered a temporary error and could not complete your request.<p>Please try again in 30 seconds.</h2>
wandb: <h2></h2>
wandb: </body></html>), retrying request


events.out.tfevents.1744849394.e85c067a8863.1094.0:   0%|          | 0.00/6.75k [00:00<?, ?B/s]

***** train metrics *****
  epoch                    =        3.0
  total_flos               =   191360GF
  train_loss               =     0.3556
  train_runtime            = 0:10:26.06
  train_samples_per_second =     17.576
  train_steps_per_second   =      2.199


# SUBIMOS EL MODELO ENTRENADO A HUGGING FACE

In [17]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/JavierCh86/distilrogerta-jchacnamadatag3/commit/14b7ab1c7769f918201325ddd1d90c082ba5c982', commit_message='End of training', commit_description='', oid='14b7ab1c7769f918201325ddd1d90c082ba5c982', pr_url=None, repo_url=RepoUrl('https://huggingface.co/JavierCh86/distilrogerta-jchacnamadatag3', endpoint='https://huggingface.co', repo_type='model', repo_id='JavierCh86/distilrogerta-jchacnamadatag3'), pr_revision=None, pr_num=None)

# PASO 6 - EVALUAMOS EL MODELO

In [18]:
metrics = trainer.evaluate(prepared_ds["validation"])
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)

***** eval metrics *****
  epoch                   =        3.0
  eval_accuracy           =     0.8333
  eval_f1                 =     0.8811
  eval_loss               =     0.5852
  eval_runtime            = 0:00:01.41
  eval_samples_per_second =    287.415
  eval_steps_per_second   =     35.927


In [19]:
from transformers import pipeline

# Asegúrate de que estos mapeos estén definidos en el modelo
model.config.id2label = {0: "not_equivalent", 1: "equivalent"}
model.config.label2id = {"not_equivalent": 0, "equivalent": 1}

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)
classifier("The course was amazing,The level of this course was amazing")


Device set to use cuda:0


[{'label': 'equivalent', 'score': 0.9876874089241028}]